In [5]:
import json, os, glob, tqdm
from PIL import Image
import numpy as np
import subprocess

def sort_by_frame(path_list):
    frame_anno = []
    for p in path_list:
        frame_idx = os.path.splitext(p.split('/')[-1].split('_')[-1])[0][5:]   # 0-4 is "frame", so we used [5:] here
        frame_anno.append(int(frame_idx))
    sorted_idx = np.argsort(frame_anno)
    sorted_path_list = []
    for idx in sorted_idx:
      sorted_path_list.append(path_list[idx])
    return sorted_path_list

In [23]:
ax = 2
c = "max"
method_json = f"/home/mint/Dev/DiFaReli/difareli-faster/visualize_scripts/TPAMI/main_results/FFHQ_CastShadows/RotateSH_figures/degrade_cmp/ffhq_rotateSH_degrade_cmp_{c}C.json"
# samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/selected_rotate_RT_60065.json"
samples = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/DiFaReli++/selected_rotate_RT.json"
sampling_rate = 2
def reduce_sampling_rate(frame_list, sampling_rate=1):
    # Reduce sampling rate and include first and last
    reduced_frame_list = frame_list[::sampling_rate]  # Take every nth frame
    if frame_list[0] not in reduced_frame_list:
        print("Adding first frame...")
        reduced_frame_list.insert(0, frame_list[0])  # Ensure the first frame is included
    if frame_list[-1] not in reduced_frame_list:
        print("Adding last frame...")
        reduced_frame_list.append(frame_list[-1])  # Ensure the last frame is included
    return reduced_frame_list

# The results were generated following the order of 
method = [f"ours_256_DiFaReli++_rotate_{c}C", 
          f"ours_difareli++_oneshot_rotate_{c}C", 
          f"ours_256_DiFaReli", 
        ]
os.makedirs("./vids/", exist_ok=True)
os.makedirs(f'./vids/all_outputs/res/', exist_ok=True)
os.makedirs(f'./vids/all_outputs/misc/', exist_ok=True)

with open(method_json, 'r') as f:
    method_json = json.load(f)

with open(samples, 'r') as f:
    samples = json.load(f)['pair']

for sample_id, sample in tqdm.tqdm(samples.items()):
    src = sample['src']
    dst = sample['dst']
    for i, m in enumerate(method):
        img_dir = method_json[m]['img_dir']
        n_frames = method_json[m]['n_frame']
        os.makedirs(f'./vids/c={c}/{src}_{dst}/{m}/', exist_ok=True)
        img_path = f'{img_dir}/src={src}/dst={dst}/Lerp_1000/n_frames={n_frames}/'
        relit = sort_by_frame(glob.glob(f'{img_path}/res_f*.png'))[1:]
        
        # Copy all images to a new folder
        for img in relit:
            ii = int(img.split('/')[-1].split('frame')[-1].split('.')[0])
            os.system(f'cp {img} ./vids/c={c}/{src}_{dst}/{m}/res_frame_{ii:04d}.png')
    
    for i, m in enumerate(method):
        # cmd = f"ffmpeg -r 24 -i ./vids/c={c}/{src}_{dst}/{m}/res_frame_%04d.png libx264 -pix_fmt yuv420p -crf 17 -y ./vids/c={c}/{src}_{dst}/{m}.mp4"
        cmd = f'ffmpeg -r 24 -i ./vids/c={c}/{src}_{dst}/{m}/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c={c}/{src}_{dst}/{m}.mp4'
        try:
            subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except subprocess.CalledProcessError:
            print(f"Error in {cmd}")
    # Concat to vstack
    filter_stack = f"-filter_complex \"[0:v][1:v][2:v]hstack=inputs=3\""
    cmd = f"""
        ffmpeg 
            -i ./vids/c={c}/{src}_{dst}/{method[2]}.mp4
            -i ./vids/c={c}/{src}_{dst}/{method[0]}.mp4
            -i ./vids/c={c}/{src}_{dst}/{method[1]}.mp4 
            {filter_stack} 
            -y ./vids/all_outputs/res/{src}_{dst}_c={c}.mp4
    """
    # Remove extra spaces and newlines
    cmd = " ".join(cmd.strip().split())
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except subprocess.CalledProcessError:
        print(f"Error in {cmd}")
    # assert False

 83%|████████▎ | 177/214 [05:25<01:02,  1.70s/it]

Error in ffmpeg -r 24 -i ./vids/c=src/63621.jpg_65924.jpg/ours_256_DiFaReli/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c=src/63621.jpg_65924.jpg/ours_256_DiFaReli.mp4
Error in ffmpeg -i ./vids/c=src/63621.jpg_65924.jpg/ours_256_DiFaReli.mp4 -i ./vids/c=src/63621.jpg_65924.jpg/ours_256_DiFaReli++_rotate_srcC.mp4 -i ./vids/c=src/63621.jpg_65924.jpg/ours_difareli++_oneshot_rotate_srcC.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" -y ./vids/all_outputs/res/63621.jpg_65924.jpg_c=src.mp4


 83%|████████▎ | 178/214 [05:26<00:54,  1.51s/it]

Error in ffmpeg -r 24 -i ./vids/c=src/63441.jpg_64533.jpg/ours_256_DiFaReli/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c=src/63441.jpg_64533.jpg/ours_256_DiFaReli.mp4
Error in ffmpeg -i ./vids/c=src/63441.jpg_64533.jpg/ours_256_DiFaReli.mp4 -i ./vids/c=src/63441.jpg_64533.jpg/ours_256_DiFaReli++_rotate_srcC.mp4 -i ./vids/c=src/63441.jpg_64533.jpg/ours_difareli++_oneshot_rotate_srcC.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" -y ./vids/all_outputs/res/63441.jpg_64533.jpg_c=src.mp4


 84%|████████▎ | 179/214 [05:27<00:48,  1.37s/it]

...

Error in ffmpeg -r 24 -i ./vids/c=src/64900.jpg_69988.jpg/ours_256_DiFaReli/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c=src/64900.jpg_69988.jpg/ours_256_DiFaReli.mp4
Error in ffmpeg -i ./vids/c=src/64900.jpg_69988.jpg/ours_256_DiFaReli.mp4 -i ./vids/c=src/64900.jpg_69988.jpg/ours_256_DiFaReli++_rotate_srcC.mp4 -i ./vids/c=src/64900.jpg_69988.jpg/ours_difareli++_oneshot_rotate_srcC.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" -y ./vids/all_outputs/res/64900.jpg_69988.jpg_c=src.mp4


100%|█████████▉| 213/214 [06:05<00:01,  1.08s/it]

Error in ffmpeg -r 24 -i ./vids/c=src/67262.jpg_69318.jpg/ours_256_DiFaReli/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c=src/67262.jpg_69318.jpg/ours_256_DiFaReli.mp4
Error in ffmpeg -i ./vids/c=src/67262.jpg_69318.jpg/ours_256_DiFaReli.mp4 -i ./vids/c=src/67262.jpg_69318.jpg/ours_256_DiFaReli++_rotate_srcC.mp4 -i ./vids/c=src/67262.jpg_69318.jpg/ours_difareli++_oneshot_rotate_srcC.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" -y ./vids/all_outputs/res/67262.jpg_69318.jpg_c=src.mp4


100%|██████████| 214/214 [06:06<00:00,  1.71s/it]

Error in ffmpeg -r 24 -i ./vids/c=src/63368.jpg_68416.jpg/ours_256_DiFaReli/res_frame_%04d.png -c:v libx264 -crf 17 -pix_fmt yuv420p -y ./vids/c=src/63368.jpg_68416.jpg/ours_256_DiFaReli.mp4
Error in ffmpeg -i ./vids/c=src/63368.jpg_68416.jpg/ours_256_DiFaReli.mp4 -i ./vids/c=src/63368.jpg_68416.jpg/ours_256_DiFaReli++_rotate_srcC.mp4 -i ./vids/c=src/63368.jpg_68416.jpg/ours_difareli++_oneshot_rotate_srcC.mp4 -filter_complex "[0:v][1:v][2:v]hstack=inputs=3" -y ./vids/all_outputs/res/63368.jpg_68416.jpg_c=src.mp4
